# Glucose × HRV Analysis

Pulls continuous glucose (CGM) and overnight HRV for one study subject from BigQuery, **interpolates HRV onto the dense glucose timeline** (`queries/glucose_hrv_interpolated.sql`), and explores the relationship.

> **Read HRV with care:** Garmin measures HRV only during sleep, so daytime glucose rows get an HRV value *interpolated across the overnight gaps* — it's a bridge between nights, not a real waking measurement.

**Setup:** `pip install -r requirements.txt` and `gcloud auth application-default login` (see README).

## 1. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJECT = 'digitaltwin-499202'
USER    = 'vincent'   # study subject: 'vincent', 'kevin', or 'christian'

client = bigquery.Client(project=PROJECT)
print(f'BigQuery client ready — project={PROJECT}, subject={USER}')

## 2. Load glucose + interpolated HRV
The query lives in `queries/glucose_hrv_interpolated.sql` and is parameterized by `@user`.

In [ ]:
sql = Path('queries/glucose_hrv_interpolated.sql').read_text().replace('PROJECT_PLACEHOLDER', PROJECT)
job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter('user', 'STRING', USER)])
df = client.query(sql, job_config=job_config).to_dataframe()
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts').reset_index(drop=True)
print(f"{len(df):,} glucose rows | {df['ts'].min()} -> {df['ts'].max()}")
df.head()

## 3. Summary statistics

In [ ]:
df[['glucose_mg_dl', 'hrv_interpolated']].describe().round(1)

## 4. Glucose over time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['ts'], df['glucose_mg_dl'], lw=0.8, color='#c0392b')
ax.axhspan(70, 140, color='#2ecc71', alpha=0.08)  # rough in-range band
ax.set_title(f'Glucose over time — {USER}')
ax.set_ylabel('Glucose (mg/dL)')
plt.tight_layout(); plt.show()

## 5. Glucose vs interpolated HRV (dual axis)

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(df['ts'], df['glucose_mg_dl'], lw=0.8, color='#c0392b')
ax1.set_ylabel('Glucose (mg/dL)', color='#c0392b')
ax2 = ax1.twinx()
ax2.plot(df['ts'], df['hrv_interpolated'], lw=0.8, color='#2980b9')
ax2.set_ylabel('HRV interpolated (ms)', color='#2980b9')
ax1.set_title(f'Glucose vs interpolated HRV — {USER}')
plt.tight_layout(); plt.show()

## 6. Relationship
Scatter + Pearson correlation. (Correlation here is dominated by the overnight interpolation, so treat it as exploratory.)

In [ ]:
sub = df.dropna(subset=['glucose_mg_dl', 'hrv_interpolated'])
r = sub['glucose_mg_dl'].corr(sub['hrv_interpolated'])
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sub['glucose_mg_dl'], sub['hrv_interpolated'], s=6, alpha=0.3)
ax.set_xlabel('Glucose (mg/dL)'); ax.set_ylabel('HRV interpolated (ms)')
ax.set_title(f'Glucose vs HRV  (Pearson r = {r:.2f}, n={len(sub):,})')
plt.tight_layout(); plt.show()
print(f'Pearson r = {r:.3f}  (n={len(sub):,})')